# CERRA(-Land) - Zarrify
***

***Author:** Javier Díez Sierra, Chus Casado Rodríguez*<br>
***Date:** 24-08-2026*<br>

**Introduction:**<br>
This notebook loads the CERRA data (total daily precipitation and 3-hourly temperature) and creates a Zarr file ready to be used in the computation of catchment-average time series for either CAMELS-ES or BEAVERS-ES.

The CERRA data was previously downloaded using the script `ocab.cerra.download.py`.

In [1]:
from pathlib import Path
import xarray as xr
import xclim
import logging
logger = logging.getLogger(__name__)

from ocab.cerra.utils import read_cerra
from ocab.cerra.zarrify import dataset_to_zarr_v3

## Configuration

In [2]:
# meteorology
meteo = 'CERRA'
path_meteo = Path('/home/casadoj/Data') / meteo / 'Iberia'
pet_method = 'HG85' # 'HG85': Hargreaves, 'DA02': Droogers-Allen

# Zarr configuration
zarr_store = path_meteo / f'{meteo}_1984-2025.zarr'
chunks = {
    'time': 365,
    'lat': -1, 
    'lon': -1,
}

## Data

### Precipitation

In [3]:
# read precipitation
var = 'tp'
paths = sorted((path_meteo / var / 'time').glob(f'{var}_*.nc'))
print(f'No. files: {len(paths)}')
pcp = read_cerra(paths, var=var, chunks=chunks)
pcp = pcp.rio.set_spatial_dims(x_dim='lon', y_dim='lat')

print(pcp.dims)
print(list(pcp.data_vars))
print(pcp.rio.crs)
print(pcp.rio.x_dim, pcp.rio.y_dim)

No. files: 42
FrozenMappingWarningOnValuesAccess({'lat': 181, 'lon': 291, 'time': 15097})
['totprec']
EPSG:4326
lon lat


### Temperature

In [ ]:
# read temperature
var = 't2m'
paths = sorted((path_meteo / var).glob(f'{var}_*.nc'))
print(f'No. files: {len(paths)}')
tmp = read_cerra(paths, var=var, chunks=chunks)
tmp = tmp.rio.set_spatial_dims(x_dim='lon', y_dim='lat')

print(tmp.dims)
print(tmp.rio.crs)
print(tmp.rio.x_dim, tmp.rio.y_dim)

### Merge Variables

In [ ]:
data = xr.merge([pcp, tmp], compat='no_conflicts', join='inner')
data = data.rio.set_spatial_dims(x_dim='lon', y_dim='lat')

print(data.dims)
print('Variables: {0}'.format(list(data.data_vars)))
print(data.rio.crs)
print(data.rio.x_dim, data.rio.y_dim)

### Potential Evapotranspiration

In [ ]:
if pet_method == 'HG85':
    pet = xclim.indices.potential_evapotranspiration(
        tasmin=data['mintemp'], 
        tasmax=data['maxtemp'], 
        lat=data['lat'],
        method=pet_method
    )
elif pet_method == 'DA02':
    pet = xclim.indices.potential_evapotranspiration(
        tasmin=data['mintemp'], 
        tasmax=data['maxtemp'], 
        pr=data['totprec'],
        lat=data['lat'],
        method=pet_method
        )
else:
    logger.error(f"`pet_method` must be either 'HG85' or 'DA02'; {pet_method} was provided.")

# convert from kg m-2 s-1 to mm/d
pet *= 86400
pet.attrs['units'] = 'mm/d'
pet.attrs['long_name'] = f'Potential Evapotranspiration ({pet_method})'

# add to dataset
data['e0'] = pet

In [ ]:
# # ensure CRS and spatial dimensions
# data = data.rio.set_spatial_dims(x_dim='lon', y_dim='lat')
# # for var in data.data_vars:
# #     data[var].rio.write_crs(data.rio.crs, inplace=True)
# #     data[var].rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
# #     print(var, da.rio.crs, da.rio.x_dim, da.rio.y_dim, sep='\t')
# for var, da in data.items():
#     da = da.rio.write_crs(data.rio.crs)
#     da = da.rio.set_spatial_dims(x_dim='lon', y_dim='lat')
#     print(var, da.rio.crs, da.rio.x_dim, da.rio.y_dim, sep='\t')
#     data[var] = da
# print('\ndataset', data.rio.crs, data.rio.x_dim, data.rio.y_dim, sep='\t')

## Export Zarr

In [ ]:
dataset_to_zarr_v3(
    ds=data,
    target_store=str(zarr_store),
    chunks=(365, -1, -1),
    shards=(365, -1, -1),
    overwrite=True
)